# P2 — FinScope Match (cell-donor) + Servicing Construction

Attach FinScope 2019 financial-inclusion flags to the NIDS backbone via a **simple cell-donor
match** (income quintile × province), then construct `monthly_trad_repayment` from the matched
product mix + external rate table. Flags are categorical (no deflation); all money stays 2017 Rands.
See [`../scratchpad/data_fusion.md`](../scratchpad/data_fusion.md).

## 1. Setup & load backbone

In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path
%matplotlib inline
import matplotlib.pyplot as plt
np.random.seed(42)

def find_root(start=Path.cwd()):
    for d in [start, *start.parents]:
        if (d / "data" / "processed" / "nids_backbone.parquet").exists():
            return d
    raise FileNotFoundError("run P0 first (nids_backbone.parquet missing)")

ROOT = find_root()
PROC = ROOT / "data" / "processed"
bb = pd.read_parquet(PROC / "nids_backbone.parquet")
summary = json.loads((PROC / "nids_backbone_summary.json").read_text())
bounds = summary["pc_income_quintile_bounds"]
edges = [-np.inf] + bounds + [np.inf]
labels = [f"Q{i}" for i in range(1, 6)]
print("backbone:", len(bb), "| province na:", int(bb.province.isna().sum()))

backbone: 10841 | province na: 0


## 2. Load FinScope donor & assign matching cells

In [2]:
fs_cols = ["HH_WEIGHT16", "Province", "Number_in_HH", "M13_MHI_Imputed", "F1", "G1", "G5", "K7",
           "G10", "G11", "G12", "G13", "G14", "G15"]
fs = pd.read_csv(ROOT / "data/raw/FINMARK_2019/Finscope South Africa 2019.csv", usecols=fs_cols)
fs = fs[fs.HH_WEIGHT16 > 0].copy()

MID = {"No Income": 0, "R1 - R999": 500, "R1 000 - R2 999": 2000, "R3 000 - R7 999": 5500,
       "R8 000 - R11 999": 10000, "R12 000 - R29 999": 21000, "R30 000 or more": 40000}
fs["mhi_mid"] = fs["M13_MHI_Imputed"].map(MID)
fs["hhsize"] = pd.to_numeric(fs["Number_in_HH"], errors="coerce").replace(0, np.nan)
fs["income_pc"] = fs["mhi_mid"] / fs["hhsize"]
fs = fs.dropna(subset=["income_pc"])
fs["income_quintile"] = pd.cut(fs["income_pc"], bins=edges, labels=labels, include_lowest=True)
print("FinScope donors:", len(fs), "| province alignment gap:",
      set(bb.province.dropna()) - set(fs.Province.dropna()))

FinScope donors: 4969 | province alignment gap: set()


## 3. Derive FinScope flags

`banked` (F1), `credit_access_formal` (formal G5 source or G10–G14 product), `informal_finance`
(informal G5 source), `savings_product` (valid K7 band). National weighted rates are the validation targets.

In [3]:
FORMAL = ["Bank", "Retail store (e.g. Woolworths, Edgars etc)",
          "Micro finance institution e.g. Wonga", "Insurance company"]
INFORMAL = ["Mashonisa or loan shark", "Stokvel society, burial society, umgalelo or savings club",
            "Friends or family or household member", "Colleagues or neighbours",
            "Employer including getting an advance on your salary"]
prod_cols = ["G10", "G11", "G12", "G13", "G14"]

fs["banked"] = (fs["F1"] == "Yes")
fs["credit_access_formal"] = fs["G5"].isin(FORMAL) | (fs[prod_cols] == "Yes").any(axis=1)
fs["informal_finance"] = fs["G5"].isin(INFORMAL)
fs["savings_product"] = fs["K7"].astype(str).str.strip().str.startswith("R")

flags = ["banked", "credit_access_formal", "informal_finance", "savings_product"]
fs_marg = {c: np.average(fs[c], weights=fs.HH_WEIGHT16) * 100 for c in flags}
print("FinScope national weighted rates (%):")
for c, v in fs_marg.items(): print(f"  {c:22}: {v:.1f}")

FinScope national weighted rates (%):
  banked                : 82.4
  credit_access_formal  : 26.4
  informal_finance      : 9.1
  savings_product       : 49.9


## 4. Cell sizes (quintile × province)

In [4]:
cs = fs.groupby(["income_quintile", "Province"], observed=True).size()
print(f"q×province cells: {len(cs)} | min {cs.min()} | median {int(cs.median())} | max {cs.max()} | <20: {int((cs<20).sum())}")

q×province cells: 45 | min 30 | median 87 | max 402 | <20: 0


## 5. Cell-donor match

In [5]:
fs = fs.rename(columns={"Province": "province"})
pool_qp = {k: g for k, g in fs.groupby(["income_quintile", "province"], observed=True)}
pool_q  = {k: g for k, g in fs.groupby("income_quintile", observed=True)}
fallbacks = 0
def draw(row):
    global fallbacks
    pool = pool_qp.get((row.income_quintile, row.province))
    if pool is None or len(pool) == 0:
        pool = pool_q.get(row.income_quintile); fallbacks += 1
    return pool.sample(1, weights=pool.HH_WEIGHT16).iloc[0]

donors = bb.apply(draw, axis=1)
pop = bb.copy()
for c in flags + ["K7"] + prod_cols:
    pop[c] = donors[c].values
print(f"matched {len(pop)} households | province-fallback draws: {fallbacks}")

matched 10841 households | province-fallback draws: 0


## 6. Match diagnostics — reproduce FinScope marginals

In [6]:
rows = []
for c in flags:
    syn = np.average(pop[c], weights=pop.w5_wgt) * 100
    rows.append({"flag": c, "synthetic_%": round(syn, 1), "finscope_%": round(fs_marg[c], 1),
                 "diff_pp": round(syn - fs_marg[c], 1)})
diag = pd.DataFrame(rows).set_index("flag"); diag

,synthetic_%,finscope_%,diff_pp
flag,,,
banked,82.8,82.4,0.4
credit_access_formal,28.7,26.4,2.3
informal_finance,9.2,9.1,0.1
savings_product,51.8,49.9,1.8


## 7. Construct `monthly_trad_repayment` (with realism guards)

Amortize `D_trad` over a **product-mix-weighted** APR/term from `data/config/credit_rate_table.csv`,
with two documented guards so the stock-balance × product-term mismatch can't produce impossible
debt service:

- **Term floor** `MIN_TERM_MONTHS` — a stock balance is never amortized faster than this (stops a
  1-month "short-term loan" term from forcing near-full-balance repayment).
- **Affordability cap** `MAX_DSTI` — an NCA-style ceiling on debt-service-to-income.

**Placeholder guard:** if the rate table still has `PLACEHOLDER_` values, servicing is left blank.

In [7]:
rate_tbl = pd.read_csv(ROOT / "data/config/credit_rate_table.csv").set_index("product_class")
has_placeholder = rate_tbl.astype(str).apply(lambda s: s.str.contains("PLACEHOLDER")).any().any()
prod_map = {"G10": "store_card", "G11": "revolving_credit", "G12": "hire_purchase",
            "G13": "short_term_loan", "G14": "personal_loan"}

# --- servicing realism parameters (documented in scratchpad/data_fusion.md) ---
MIN_TERM_MONTHS = 6      # term floor: stock debt not amortized faster than 6 months
MAX_DSTI        = 0.65   # NCA-style affordability ceiling on debt service / income

if has_placeholder:
    print("⚠ credit_rate_table.csv still contains PLACEHOLDER_ values.")
    print("  -> monthly_trad_repayment NOT computed. Fill the table to enable.")
    pop["monthly_trad_repayment"] = pd.NA
else:
    apr = rate_tbl["apr_annual"].astype(float); term = rate_tbl["term_months"].astype(float)
    def servicing(row):
        if row.D_trad <= 0:
            return pd.Series({"repay": 0.0, "floored": 0})
        held = [prod_map[c] for c in prod_map if row.get(c) == "Yes"] or ["other_default"]
        a = float(np.mean([apr[c] for c in held]))
        n_raw = float(np.mean([term[c] for c in held]))
        n = max(n_raw, MIN_TERM_MONTHS)
        r = a / 12.0
        pmt = row.D_trad * r / (1 - (1 + r) ** (-n)) if r > 0 else row.D_trad / n
        return pd.Series({"repay": pmt, "floored": int(n_raw < MIN_TERM_MONTHS)})

    res = pop.apply(servicing, axis=1)
    pop["repay_uncapped"] = res["repay"]
    cap = MAX_DSTI * pop["w5_hhincome"]
    pop["monthly_trad_repayment"] = np.minimum(pop["repay_uncapped"], cap)
    pop["dsti"] = (pop["monthly_trad_repayment"] / pop["w5_hhincome"].replace(0, np.nan)).fillna(0)
    n_floored = int(res["floored"].sum()); n_capped = int((pop["repay_uncapped"] > cap).sum())
    print(f"servicing computed. term-floored (<{MIN_TERM_MONTHS}m): {n_floored} | "
          f"affordability-capped at {MAX_DSTI:.0%}: {n_capped} households")
    print("post-fix: max DSTI =", round(pop.dsti.max(), 3),
          "| households with repay>income:", int((pop.monthly_trad_repayment > pop.w5_hhincome).sum()))
    print("\nweighted mean DSTI by quintile (%):")
    print(pop.groupby("income_quintile", observed=True)
             .apply(lambda d: np.average(d.dsti, weights=d.w5_wgt) * 100).round(1).to_string())

servicing computed. term-floored (<6m): 33 | affordability-capped at 65%: 110 households
post-fix: max DSTI = 0.65 | households with repay>income: 0

weighted mean DSTI by quintile (%):
income_quintile
Q1    3.7
Q2    3.0
Q3    4.5
Q4    5.6
Q5    5.6


## 8. Persist matched population

In [8]:
out = PROC / "synthetic_population_matched.parquet"
pop.to_parquet(out, index=False)
pop.head(50).to_csv(PROC / "synthetic_population_matched_preview.csv", index=False)
diag_json = {"n_households": int(len(pop)), "province_fallback_draws": int(fallbacks),
             "match_diagnostics_pp": diag["diff_pp"].to_dict(),
             "servicing_computed": (not bool(has_placeholder))}
(PROC / "p2_match_summary.json").write_text(json.dumps(diag_json, indent=2))
print("wrote:", out.name, "| preview | p2_match_summary.json")
print(json.dumps(diag_json, indent=2))

wrote: synthetic_population_matched.parquet | preview | p2_match_summary.json
{
  "n_households": 10841,
  "province_fallback_draws": 0,
  "match_diagnostics_pp": {
    "banked": 0.4,
    "credit_access_formal": 2.3,
    "informal_finance": 0.1,
    "savings_product": 1.8
  },
  "servicing_computed": true
}


Next — **P3**: weighted resample to 5,000; **P4**: validation report.